In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import GPT2Tokenizer
import math
import copy

# ==========================================
# 1. Diffusion Utilities (Sqrt Schedule)
# ==========================================
def compute_sqrt_alpha_bar(t, T_max=2000, s=1e-4):
    """
    Sqrt noise schedule from Diffusion-LM (Appendix A).
    alpha_bar_t = 1 - sqrt(t/T + s)
    """
    # t is expected to be a tensor of shape (batch_size, 1, 1)
    t_norm = t.float() / T_max
    alpha_bar = 1.0 - torch.sqrt(t_norm + s)
    # Clamp to prevent negative variances or exact zeros
    return torch.clamp(alpha_bar, min=1e-5, max=0.9999)

class EMA:
    """Exponential Moving Average for model weights (Crucial for TRM stability)"""
    def __init__(self, model, decay=0.999):
        self.model = model
        self.decay = decay
        self.shadow = {}
        self.backup = {}
        for name, param in model.named_parameters():
            if param.requires_grad:
                self.shadow[name] = param.data.clone()

    def update(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                new_average = (1.0 - self.decay) * param.data + self.decay * self.shadow[name]
                self.shadow[name] = new_average.clone()

    def apply_shadow(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                self.backup[name] = param.data
                param.data = self.shadow[name]

    def restore(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                param.data = self.backup[name]

In [3]:
# ==========================================
# 2. Model Architecture (TRM + Diffusion)
# ==========================================
class BidirectionalTransformerLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.norm1 = nn.LayerNorm(d_model)
        
        self.mlp = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model)
        )
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        batch_size, seq_len, _ = x.size()
        q = self.q_proj(x).view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1, 2)

        attn_out = F.scaled_dot_product_attention(q, k, v, is_causal=False)
        attn_out = attn_out.transpose(1, 2).contiguous().view(batch_size, seq_len, -1)
        
        x = self.norm1(x + self.out_proj(attn_out))
        x = self.norm2(x + self.mlp(x))
        return x

class TRM_Diffusion(nn.Module):
    def __init__(self, vocab_size, d_model=512, n_heads=8, d_ff=2048, num_layers=4):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        
        # End-to-End Embeddings
        self.token_emb = nn.Embedding(vocab_size, d_model)
        # FREEZE EMBEDDINGS to force the TRM layers to learn denoising.
        # Once your loss curve looks normal and outputs are coherent, set this to True.
        self.token_emb.weight.requires_grad = True
        self.pos_emb = nn.Embedding(1024, d_model) 
        
        # Timestep embedding (Diffusion specific)
        self.time_mlp = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model)
        )
        
        # TRM Backbone (Less is More - keeping layers relatively low)
        self.layers = nn.ModuleList([
            BidirectionalTransformerLayer(d_model, n_heads, d_ff) 
            for _ in range(num_layers)
        ])
        
        # Continuous Output Head (Predicting x_0)
        self.output_head = nn.Linear(d_model, d_model)
        self.z_init = nn.Parameter(torch.randn(1, 1, d_model))

        # ==========================================
        # NEW: DISCRETE OUTPUT HEAD (Predicting words)
        # ==========================================
        # This maps the continuous prediction back to vocabulary logits
        self.lm_head = nn.Linear(d_model, vocab_size)

        # Halting Head (ACT)
        self.q_head = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.ReLU(),
            nn.Linear(d_model // 2, 1),
            nn.Sigmoid() 
        )

    def get_sinusoidal_embeddings(self, t):
        """Standard sinusoidal positional embeddings for time"""
        half_dim = self.d_model // 2
        emb = math.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=t.device) * -emb)
        emb = t.float().unsqueeze(1) * emb.unsqueeze(0)
        emb = torch.cat((emb.sin(), emb.cos()), dim=-1)
        return emb

    def latent_recursion(self, x_t_emb, z, t_emb, n):
        """Inner reasoning loop"""
        for _ in range(n):
            # Combine current noisy observation, latent thought, and time
            combined_state = x_t_emb + z + t_emb.unsqueeze(1)
            
            for layer in self.layers:
                combined_state = layer(combined_state)
            z = combined_state
            
        pred_x0_emb = self.output_head(z)
        return pred_x0_emb, z

    def deep_recursion(self, x_t_emb, z, t, n=6, T=3):
        """Deep Supervision loop mimicking pseudo-depth"""
        t_emb = self.time_mlp(self.get_sinusoidal_embeddings(t))
        
        # 1. Thinking Phase (No Gradients)
        with torch.no_grad():
            for _ in range(T - 1):
                _, z = self.latent_recursion(x_t_emb, z, t_emb, n)
                
        # 2. Action Phase (Track Gradients)
        pred_x0_emb, z = self.latent_recursion(x_t_emb, z, t_emb, n)
        
        z_mean = z.mean(dim=1) 
        q_hat = self.q_head(z_mean).squeeze(-1)
        
        return pred_x0_emb, z.detach(), q_hat

In [ ]:
@torch.no_grad()
def generate_with_prompt(model_path, prompt_text, seq_len=64, temperature=0.8):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    T_DIFFUSION = 2000
    CLAMP_START = 500 
    
    tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
    vocab_size = len(tokenizer)
    
    model = TRM_Diffusion(vocab_size=vocab_size, d_model=512, num_layers=4).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device)['model_state_dict'])
    model.eval()
    
    vocab_embeddings = model.token_emb.weight 
    positions = torch.arange(seq_len, device=device).unsqueeze(0)
    pos_embeddings = model.pos_emb(positions)
    
    # ==========================================
    # 1. PREPARE THE PROMPT
    # ==========================================
    prompt_tokens = tokenizer(prompt_text, return_tensors="pt")["input_ids"].to(device)
    prompt_len = prompt_tokens.shape[1]
    
    # Get the clean, exact embeddings for the prompt (Word + Position)
    clean_prompt_emb = model.token_emb(prompt_tokens) + pos_embeddings[:, :prompt_len, :]
    
    print(f"Prompt length: {prompt_len} tokens. Generating remaining {seq_len - prompt_len} tokens...")
    
    # Start with pure Gaussian Noise for the whole sequence
    x_t_emb = torch.randn(1, seq_len, model.d_model, device=device)
    
    for t_step in reversed(range(1, T_DIFFUSION + 1)):
        t_tensor = torch.full((1,), t_step, device=device, dtype=torch.long)
        
        # ==========================================
        # 2. FORCE THE PROMPT INTO THE NOISE
        # ==========================================
        # We calculate exactly how much noise the prompt *should* have at this timestep
        alpha_bar_t = compute_sqrt_alpha_bar(t_tensor).view(-1, 1, 1)
        prompt_noise = torch.randn_like(clean_prompt_emb)
        
        # Create the mathematically accurate noised prompt
        noised_prompt = torch.sqrt(alpha_bar_t) * clean_prompt_emb + torch.sqrt(1 - alpha_bar_t) * prompt_noise
        
        # Overwrite the beginning of the sequence with our noised prompt!
        x_t_emb[:, :prompt_len, :] = noised_prompt
        
        # ==========================================
        # 3. STANDARD TRM DENOISING (Predicts the whole sequence)
        # ==========================================
        z = model.z_init.expand(1, seq_len, -1)
        pred_x0_combined, _, _ = model.deep_recursion(x_t_emb, z, t_tensor, n=6, T=3)
        
        # --- DELAYED CLAMPING ---
        if t_step <= CLAMP_START:
            pred_word_emb = pred_x0_combined - pos_embeddings
            clamped_word_emb, token_ids = clamp_to_nearest_word(pred_word_emb, vocab_embeddings)
            final_x0 = clamped_word_emb + pos_embeddings
            
            # FORCE the prompt tokens to be exactly correct during clamping
            final_x0[:, :prompt_len, :] = clean_prompt_emb
        else:
            final_x0 = pred_x0_combined 
            final_x0[:, :prompt_len, :] = clean_prompt_emb # Keep prompt anchored
            
        # --- DDPM POSTERIOR MATH (Same as before) ---
        if t_step > 1:
            t_minus_1 = torch.full((1,), t_step - 1, device=device, dtype=torch.long)
            alpha_bar_t_minus_1 = compute_sqrt_alpha_bar(t_minus_1).view(-1, 1, 1)
            alpha_t = alpha_bar_t / alpha_bar_t_minus_1
            beta_t = 1.0 - alpha_t
            
            c1 = (torch.sqrt(alpha_bar_t_minus_1) * beta_t) / (1.0 - alpha_bar_t)
            c2 = (torch.sqrt(alpha_t) * (1.0 - alpha_bar_t_minus_1)) / (1.0 - alpha_bar_t)
            
            posterior_mean = c1 * final_x0 + c2 * x_t_emb
            posterior_variance = ((1.0 - alpha_bar_t_minus_1) / (1.0 - alpha_bar_t)) * beta_t
            
            noise = torch.randn_like(x_t_emb) * temperature
            x_t_emb = posterior_mean + torch.sqrt(posterior_variance) * noise
        else:
            x_t_emb = final_x0

    # Decode the final token IDs
    final_text = tokenizer.batch_decode(token_ids, skip_special_tokens=True)
    print("\n--- Generated Story ---")
    print(final_text[0])
    return final_text

# Example Usage:
# generate_with_prompt("trm_diffusion_epoch_9.pt", "Once upon a time, there was a little dog named")

Prompt length: 11 tokens. Generating remaining 53 tokens...

--- Generated Story ---
Once upon a time, there was a little dog named lived every he red red inside, three
 upon animals sad to down of see knewWhat started and off
 go the out parks by thingsily things held started biggers quickly scared. a shinyThat can sad knew� away heard andily.WhenBut thought


['Once upon a time, there was a little dog named lived every he red red inside, three\n upon animals sad to down of see knewWhat started and off\n go the out parks by thingsily things held started biggers quickly scared. a shinyThat can sad knew� away heard andily.WhenBut thought']

In [17]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
vocab_size = len(tokenizer)
model = TRM_Diffusion(vocab_size=vocab_size, d_model=512, num_layers=4).to(device)
model.load_state_dict(torch.load("C:/Users/MSI-1/Desktop/adnan_final/small diff-succes-modls/trm_diffusion_epoch_9.pt", map_location=device)['model_state_dict'])
params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable parameters in the model: {params/1e6:.2f}M")

Total trainable parameters in the model: 39.79M


train longer

In [4]:
# ==========================================
# 3. Dataset & Dataloader 
# ==========================================
def get_rocstories_dataloader(batch_size=64, seq_len=64):
    """Loads ROCStories and tokenizes them"""
    tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
    tokenizer.pad_token = tokenizer.eos_token
    
    # We use a subset/variant of ROCStories available on HuggingFace
    dataset = load_dataset("roneneldan/TinyStories", split="train[:1000000]") 
    
    def tokenize(batch):
        # TinyStories just uses the "text" column
        return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=seq_len, return_tensors="pt")
    
    dataset = dataset.map(tokenize, batched=True, remove_columns=dataset.column_names)
    dataset.set_format(type="torch", columns=["input_ids"])
    return DataLoader(dataset, batch_size=batch_size, shuffle=True), len(tokenizer)

In [ ]:
import os
import torch
import torch.nn.functional as F
import wandb
from tqdm import tqdm # Import tqdm here!

# ==========================================
# 4. Training Loop 
# ==========================================
def train():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    seq_len = 256
    batch_size = 128
    T_DIFFUSION = 2000
    EPOCHS = 10 # 1 Million stories = ~937,500 total steps over 15 epochs
    
    N_RECURSIONS = 6
    T_CYCLES = 3 
    
    dataloader, vocab_size = get_rocstories_dataloader(batch_size, seq_len)
    
    model = TRM_Diffusion(vocab_size=vocab_size, d_model=512, num_layers=4).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)
    ema = EMA(model, decay=0.999)
    
    wandb.init(
        project="trm-continuous-diffusion",
        name="1M-stories-run",
        config={
            "dataset_size": "1M",
            "epochs": EPOCHS,
            "batch_size": batch_size,
            "seq_len": seq_len,
            "learning_rate": 1e-4
        }
    )
    
    print("Starting Training...")
    model.train()
    global_step = 0 
    
    for epoch in range(EPOCHS): 
        # Wrap the dataloader in tqdm to create the progress bar
        pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=True)
        
        for step, batch in enumerate(pbar):
            input_ids = batch["input_ids"].to(device)
            bs = input_ids.shape[0]
            
            # 1. Clean x_0 Embeddings
            y_true_emb = model.token_emb(input_ids)
            positions = torch.arange(seq_len, device=device).unsqueeze(0).expand(bs, seq_len)
            y_true_emb = y_true_emb + model.pos_emb(positions)
            
            # 2. Sample random timestep t and apply Sqrt Noise
            t = torch.randint(1, T_DIFFUSION + 1, (bs,), device=device)
            alpha_bar_t = compute_sqrt_alpha_bar(t, T_max=T_DIFFUSION).view(bs, 1, 1)
            noise = torch.randn_like(y_true_emb)
            
            # Forward Process q(x_t | x_0)
            x_t_emb = torch.sqrt(alpha_bar_t) * y_true_emb + torch.sqrt(1 - alpha_bar_t) * noise
            
            # 3. TRM Denoiser Prediction
            z = model.z_init.expand(bs, seq_len, -1)
            pred_x0_emb, z, _ = model.deep_recursion(x_t_emb, z, t, n=N_RECURSIONS, T=T_CYCLES)
            
            # 4. Calculate Diffusion-LM Losses
            loss_recon = F.mse_loss(pred_x0_emb, y_true_emb)
            loss_emb_anchor = F.mse_loss(y_true_emb, pred_x0_emb.detach())
            total_loss = loss_recon + (0.1 * loss_emb_anchor)
            
            # 5. Backpropagate
            optimizer.zero_grad()
            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            ema.update()
            
            # ==========================================
            # UPDATE METRICS LIVE
            # ==========================================
            wandb.log({
                "train/total_loss": total_loss.item(),
                "train/loss_recon": loss_recon.item(),
                "epoch": epoch,
                "global_step": global_step
            })
            
            # Updates the right side of the progress bar with the current loss
            if step % 10 == 0:
                pbar.set_postfix({'Loss': f"{total_loss.item():.4f}"})
                
            # Mid-epoch save
            if global_step > 0 and global_step % 10000 == 0:
                ema.apply_shadow()
                save_path = f"trm_diffusion_step_{global_step}.pt"
                torch.save({
                    'epoch': epoch,
                    'global_step': global_step,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                }, save_path)
                wandb.save(save_path) 
                ema.restore()
                
            global_step += 1

        print(f"--- Epoch {epoch+1} Complete ---")
        ema.apply_shadow()
        save_path = f"trm_diffusion_epoch_{epoch+1}_complete.pt"
        torch.save({
            'epoch': epoch,
            'global_step': global_step,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
        }, save_path)
        wandb.save(save_path) 
        ema.restore()
        
    wandb.finish()

if __name__ == "__main__":
    train()

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.
Epoch 3/10: 100%|██████████| 7813/7813 [2:33:06<00:00,  1.18s/it, Loss=0.0377]  


--- Epoch 3 Complete ---


wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.
Epoch 4/10: 100%|██████████| 7813/7813 [2:32:50<00:00,  1.17s/it, Loss=0.0106]  


--- Epoch 4 Complete ---


wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.
Epoch 5/10:  33%|███▎      | 2552/7813 [48:34<1:40:13,  1.14s/it, Loss=0.0081]

In [4]:
import os
import torch
import torch.nn.functional as F
import wandb
from tqdm import tqdm

# ==========================================
# 4. Training Loop (Resumable)
# ==========================================
def train():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    seq_len = 256
    batch_size = 128
    T_DIFFUSION = 2000
    EPOCHS = 10 
    
    N_RECURSIONS = 6
    T_CYCLES = 3 
    
    # --- RESUME CONFIGURATION ---
    resume_training = True
    checkpoint_path = "trm_diffusion_epoch_4_complete.pt"
    # ----------------------------

    dataloader, vocab_size = get_rocstories_dataloader(batch_size, seq_len)
    
    model = TRM_Diffusion(vocab_size=vocab_size, d_model=512, num_layers=4).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)
    
    start_epoch = 0
    global_step = 0

    # ==========================================
    # LOAD CHECKPOINT LOGIC
    # ==========================================
    if resume_training and os.path.exists(checkpoint_path):
        print(f"Loading checkpoint: {checkpoint_path}")
        checkpoint = torch.load(checkpoint_path, map_location=device)
        
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        
        # If epoch 3 (which is Epoch 4 for humans) finished, start at 4 (Epoch 5)
        start_epoch = checkpoint['epoch'] + 1 
        global_step = checkpoint['global_step']
        
        print(f"Successfully resumed! Starting at Epoch {start_epoch + 1}, Global Step {global_step}")
    else:
        print("Starting Training from scratch...")

    # Initialize EMA *after* loading the model weights so it tracks the correct values
    ema = EMA(model, decay=0.999)
    
    wandb.init(
        project="trm-continuous-diffusion",
        name="1M-stories-run-resumed",
        config={
            "dataset_size": "1M",
            "epochs": EPOCHS,
            "batch_size": batch_size,
            "seq_len": seq_len,
            "learning_rate": 1e-4,
            "resumed_from": checkpoint_path
        }
    )
    
    model.train()
    
    # Modify the range to start from 'start_epoch' instead of 0
    for epoch in range(start_epoch, EPOCHS): 
        pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=True)
        
        for step, batch in enumerate(pbar):
            input_ids = batch["input_ids"].to(device)
            bs = input_ids.shape[0]
            
            # 1. Clean x_0 Embeddings
            y_true_emb = model.token_emb(input_ids)
            positions = torch.arange(seq_len, device=device).unsqueeze(0).expand(bs, seq_len)
            y_true_emb = y_true_emb + model.pos_emb(positions)
            
            # 2. Sample random timestep t and apply Sqrt Noise
            t = torch.randint(1, T_DIFFUSION + 1, (bs,), device=device)
            alpha_bar_t = compute_sqrt_alpha_bar(t, T_max=T_DIFFUSION).view(bs, 1, 1)
            noise = torch.randn_like(y_true_emb)
            
            # Forward Process q(x_t | x_0)
            x_t_emb = torch.sqrt(alpha_bar_t) * y_true_emb + torch.sqrt(1 - alpha_bar_t) * noise
            
            # 3. TRM Denoiser Prediction
            z = model.z_init.expand(bs, seq_len, -1)
            pred_x0_emb, z, _ = model.deep_recursion(x_t_emb, z, t, n=N_RECURSIONS, T=T_CYCLES)
            
            # 4. Calculate Diffusion-LM Losses
            loss_recon = F.mse_loss(pred_x0_emb, y_true_emb)
            loss_emb_anchor = F.mse_loss(y_true_emb, pred_x0_emb.detach())
            total_loss = loss_recon + (0.1 * loss_emb_anchor)
            
            # 5. Backpropagate
            optimizer.zero_grad()
            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            ema.update()
            
            # ==========================================
            # UPDATE METRICS LIVE
            # ==========================================
            wandb.log({
                "train/total_loss": total_loss.item(),
                "train/loss_recon": loss_recon.item(),
                "epoch": epoch,
                "global_step": global_step
            })
            
            if step % 10 == 0:
                pbar.set_postfix({'Loss': f"{total_loss.item():.4f}"})
                
            # Mid-epoch save
            if global_step > 0 and global_step % 10000 == 0:
                ema.apply_shadow()
                save_path = f"trm_diffusion_step_{global_step}.pt"
                torch.save({
                    'epoch': epoch,
                    'global_step': global_step,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                }, save_path)
                wandb.save(save_path) 
                ema.restore()
                
            global_step += 1

        print(f"--- Epoch {epoch+1} Complete ---")
        ema.apply_shadow()
        save_path = f"trm_diffusion_epoch_{epoch+1}_complete.pt"
        torch.save({
            'epoch': epoch,
            'global_step': global_step,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
        }, save_path)
        wandb.save(save_path) 
        ema.restore()
        
    wandb.finish()

if __name__ == "__main__":
    train()

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: rao-adnan-098 (rao-adnan-098-jamia-millia-islamia) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch 5/10:  71%|███████   | 5521/7813 [1:49:57<45:38,  1.20s/it, Loss=0.0075]  
socket.send() raised exception.


KeyboardInterrupt: 

Error in callback <bound method _WandbInit._post_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x7e9fcae82bd0>> (for post_run_cell), with arguments args (<ExecutionResult object at 7ea18b723190, execution_count=4 error_before_exec=None error_in_exec= info=<ExecutionInfo object at 7ea18b71c750, raw_cell="import os
import torch
import torch.nn.functional .." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://ssh-remote%2Brunpod_1/workspace/Diffusion_TRM/continuous-diff.ipynb#X16sdnNjb2RlLXJlbW90ZQ%3D%3D> result=None>,),kwargs {}:


ConnectionResetError: Connection lost

Updated the Loss objective

In [8]:
import os
import torch
import torch.nn.functional as F
import wandb
from tqdm import tqdm # Import tqdm here!

# ==========================================
# 4. Training Loop 
# ==========================================
def train():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    seq_len = 256
    batch_size = 128
    T_DIFFUSION = 2000
    EPOCHS = 4 # 1 Million stories = ~937,500 total steps over 15 epochs
    
    N_RECURSIONS = 2
    T_CYCLES = 2
    
    dataloader, vocab_size = get_rocstories_dataloader(batch_size, seq_len)
    
    model = TRM_Diffusion(vocab_size=vocab_size, d_model=512, num_layers=4).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)
    ema = EMA(model, decay=0.999)
    
    wandb.init(
        project="trm-continuous-diffusion",
        name="1M-stories-run",
        config={
            "dataset_size": "1M",
            "epochs": EPOCHS,
            "batch_size": batch_size,
            "seq_len": seq_len,
            "learning_rate": 1e-4
        }
    )
    
    print("Starting Training...")
    model.train()
    global_step = 0 
    
    for epoch in range(EPOCHS): 
        # Wrap the dataloader in tqdm to create the progress bar
        pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=True)
        
        for step, batch in enumerate(pbar):
            input_ids = batch["input_ids"].to(device)
            bs = input_ids.shape[0]
            
            # 1. Clean x_0 Embeddings
            y_true_emb = model.token_emb(input_ids)
            positions = torch.arange(seq_len, device=device).unsqueeze(0).expand(bs, seq_len)
            y_true_emb = y_true_emb + model.pos_emb(positions)
            
            # 2. Sample random timestep t and apply Sqrt Noise
            t = torch.randint(1, T_DIFFUSION + 1, (bs,), device=device)
            alpha_bar_t = compute_sqrt_alpha_bar(t, T_max=T_DIFFUSION).view(bs, 1, 1)
            noise = torch.randn_like(y_true_emb)
            
            # Forward Process q(x_t | x_0)
            x_t_emb = torch.sqrt(alpha_bar_t) * y_true_emb + torch.sqrt(1 - alpha_bar_t) * noise
            
            # 3. TRM Denoiser Prediction
            z = model.z_init.expand(bs, seq_len, -1)
            pred_x0_emb, z, _ = model.deep_recursion(x_t_emb, z, t, n=N_RECURSIONS, T=T_CYCLES)

            # ==========================================
            # NEW: PROJECT TO DISCRETE VOCABULARY
            # ==========================================
            logits = model.lm_head(pred_x0_emb) # Shape: (bs, seq_len, vocab_size)
            
            # 4. Calculate Diffusion-LM Losses
            # Loss A: Continuous MSE (Denoising the vector)
            loss_recon = F.mse_loss(pred_x0_emb, y_true_emb)

            # Loss B: Discrete Cross-Entropy (Anchoring the vector to a word)
            # Flattening logits and input_ids for standard PyTorch CE loss
            loss_ce = F.cross_entropy(logits.view(-1, vocab_size), input_ids.view(-1))

            # Total Loss: Combine MSE and CE. 
            # You may need to tune the 0.5 weight, but this prevents embedding collapse.
            total_loss = loss_recon + (0.5 * loss_ce)
            
            # 5. Backpropagate
            optimizer.zero_grad()
            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            ema.update()
            
            # ==========================================
            # UPDATE METRICS LIVE
            # ==========================================
            wandb.log({
                "train/total_loss": total_loss.item(),
                "train/loss_recon": loss_recon.item(),
                "train/loss_ce_discrete": loss_ce.item(),
                "epoch": epoch,
                "global_step": global_step
            })
            
            # Updates the right side of the progress bar with the current loss
            if step % 10 == 0:
                pbar.set_postfix({'Loss': f"{total_loss.item():.4f}"})
                
            # Mid-epoch save
            if global_step > 0 and global_step % 10000 == 0:
                ema.apply_shadow()
                save_path = f"trm_diffusion_step_{global_step}.pt"
                torch.save({
                    'epoch': epoch,
                    'global_step': global_step,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                }, save_path)
                wandb.save(save_path) 
                ema.restore()
                
            global_step += 1

        print(f"--- Epoch {epoch+1} Complete ---")
        ema.apply_shadow()
        save_path = f"trm_diffusion_epoch_{epoch+1}_complete.pt"
        torch.save({
            'epoch': epoch,
            'global_step': global_step,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
        }, save_path)
        wandb.save(save_path) 
        ema.restore()
        
    wandb.finish()

if __name__ == "__main__":
    train()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: rao-adnan-098 (rao-adnan-098-jamia-millia-islamia) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Starting Training...


Epoch 1/4: 100%|██████████| 7813/7813 [1:10:01<00:00,  1.86it/s, Loss=0.3170]


--- Epoch 1 Complete ---


wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.
Epoch 2/4:   0%|          | 8/7813 [00:04<1:20:05,  1.62it/s, Loss=0.4630]


KeyboardInterrupt: 

Error in callback <bound method _WandbInit._post_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x72ba01ae7910>> (for post_run_cell), with arguments args (<ExecutionResult object at 72ba01a82790, execution_count=8 error_before_exec=None error_in_exec= info=<ExecutionInfo object at 72ba01aa9f10, raw_cell="import os
import torch
import torch.nn.functional .." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://ssh-remote%2Brunpod_1/workspace/Diffusion_TRM/continuous-diff.ipynb#X22sdnNjb2RlLXJlbW90ZQ%3D%3D> result=None>,),kwargs {}:


ConnectionResetError: Connection lost

In [5]:
import os
import torch
import torch.nn.functional as F
import wandb
from tqdm import tqdm

# ==========================================
# 4. Training Loop (Resumable)
# ==========================================
def train():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    seq_len = 256
    batch_size = 128
    T_DIFFUSION = 2000
    EPOCHS = 10 
    
    N_RECURSIONS = 2
    T_CYCLES = 2 
    
    # --- RESUME CONFIGURATION ---
    resume_training = True
    checkpoint_path = "trm_diffusion_epoch_1_complete.pt"
    # ----------------------------

    dataloader, vocab_size = get_rocstories_dataloader(batch_size, seq_len)
    
    model = TRM_Diffusion(vocab_size=vocab_size, d_model=512, num_layers=4).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.01)
    
    start_epoch = 0
    global_step = 0

    # ==========================================
    # LOAD CHECKPOINT LOGIC
    # ==========================================
    if resume_training and os.path.exists(checkpoint_path):
        print(f"Loading checkpoint: {checkpoint_path}")
        checkpoint = torch.load(checkpoint_path, map_location=device)
        
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        
        # If epoch 3 (which is Epoch 4 for humans) finished, start at 4 (Epoch 5)
        start_epoch = checkpoint['epoch'] + 1 
        global_step = checkpoint['global_step']
        
        print(f"Successfully resumed! Starting at Epoch {start_epoch + 1}, Global Step {global_step}")
    else:
        print("Starting Training from scratch...")

    # Initialize EMA *after* loading the model weights so it tracks the correct values
    ema = EMA(model, decay=0.999)
    
    wandb.init(
        project="trm-continuous-diffusion",
        name="1M-stories-run-resumed",
        config={
            "dataset_size": "1M",
            "epochs": EPOCHS,
            "batch_size": batch_size,
            "seq_len": seq_len,
            "learning_rate": 5e-5,
            "resumed_from": checkpoint_path
        }
    )
    
    model.train()
    
    # Modify the range to start from 'start_epoch' instead of 0
    for epoch in range(start_epoch, EPOCHS): 
        pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=True)
        
        for step, batch in enumerate(pbar):
            input_ids = batch["input_ids"].to(device)
            bs = input_ids.shape[0]
            
            # 1. Clean x_0 Embeddings
            y_true_emb = model.token_emb(input_ids)
            positions = torch.arange(seq_len, device=device).unsqueeze(0).expand(bs, seq_len)
            y_true_emb = y_true_emb + model.pos_emb(positions)
            
            # 2. Sample random timestep t and apply Sqrt Noise
            t = torch.randint(1, T_DIFFUSION + 1, (bs,), device=device)
            alpha_bar_t = compute_sqrt_alpha_bar(t, T_max=T_DIFFUSION).view(bs, 1, 1)
            noise = torch.randn_like(y_true_emb)
            
            # Forward Process q(x_t | x_0)
            x_t_emb = torch.sqrt(alpha_bar_t) * y_true_emb + torch.sqrt(1 - alpha_bar_t) * noise
            
            # 3. TRM Denoiser Prediction
            z = model.z_init.expand(bs, seq_len, -1)
            pred_x0_emb, z, _ = model.deep_recursion(x_t_emb, z, t, n=N_RECURSIONS, T=T_CYCLES)

            # ==========================================
            # NEW: PROJECT TO DISCRETE VOCABULARY
            # ==========================================
            logits = model.lm_head(pred_x0_emb) # Shape: (bs, seq_len, vocab_size)
            
            # 4. Calculate Diffusion-LM Losses
            # Loss A: Continuous MSE (Denoising the vector)
            loss_recon = F.mse_loss(pred_x0_emb, y_true_emb)

            # Loss B: Discrete Cross-Entropy (Anchoring the vector to a word)
            # Flattening logits and input_ids for standard PyTorch CE loss
            loss_ce = F.cross_entropy(logits.view(-1, vocab_size), input_ids.view(-1))

            # Total Loss: Combine MSE and CE. 
            # You may need to tune the 0.5 weight, but this prevents embedding collapse.
            total_loss = loss_recon + (0.5 * loss_ce)
            
            # 5. Backpropagate
            optimizer.zero_grad()
            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            ema.update()
            
            # ==========================================
            # UPDATE METRICS LIVE
            # ==========================================
            wandb.log({
                "train/total_loss": total_loss.item(),
                "train/loss_recon": loss_recon.item(),
                "train/loss_ce_discrete": loss_ce.item(),
                "epoch": epoch,
                "global_step": global_step
            })
            
            if step % 10 == 0:
                pbar.set_postfix({'Loss': f"{total_loss.item():.4f}"})
                
            # Mid-epoch save
            if global_step > 0 and global_step % 10000 == 0:
                ema.apply_shadow()
                save_path = f"trm_diffusion_step_{global_step}.pt"
                torch.save({
                    'epoch': epoch,
                    'global_step': global_step,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                }, save_path)
                wandb.save(save_path) 
                ema.restore()
                
            global_step += 1

        print(f"--- Epoch {epoch+1} Complete ---")
        ema.apply_shadow()
        save_path = f"trm_diffusion_epoch_{epoch+1}_complete.pt"
        torch.save({
            'epoch': epoch,
            'global_step': global_step,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
        }, save_path)
        wandb.save(save_path) 
        ema.restore()
        
    wandb.finish()

if __name__ == "__main__":
    train()

Loading checkpoint: trm_diffusion_epoch_1_complete.pt


/tmp/ipykernel_7254/4067438372.py:38: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path, map_location=device)
wandb: [wandb.login()] Load

Successfully resumed! Starting at Epoch 2, Global Step 7813


Epoch 2/10: 100%|██████████| 7813/7813 [1:11:07<00:00,  1.83it/s, Loss=0.2384]


--- Epoch 2 Complete ---


...

--- Epoch 8 Complete ---


wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.
Epoch 9/10:   3%|▎         | 207/7813 [01:53<1:09:33,  1.82it/s, Loss=0.1072]


KeyboardInterrupt: 

Error in callback <bound method _WandbInit._post_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x76abd4712d50>> (for post_run_cell), with arguments args (<ExecutionResult object at 76adacc56450, execution_count=5 error_before_exec=None error_in_exec= info=<ExecutionInfo object at 76adac3be2d0, raw_cell="import os
import torch
import torch.nn.functional .." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://ssh-remote%2Brunpod_1/workspace/Diffusion_TRM/continuous-diff.ipynb#X13sdnNjb2RlLXJlbW90ZQ%3D%3D> result=None>,),kwargs {}:


ConnectionResetError: Connection lost

In [10]:
import sys
!{sys.executable} -m pip install tensorboard

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
You should consider upgrading via the 'c:\Users\MSI-1\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.


In [11]:
import torch
import torch.nn.functional as F
from tqdm import tqdm
from torch.utils.tensorboard import SummaryWriter # <-- Built-in PyTorch logger

def train():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    seq_len = 128
    batch_size = 16
    T_DIFFUSION = 2000
    EPOCHS = 10 
    
    N_RECURSIONS = 6
    T_CYCLES = 3 
    
    dataloader, vocab_size = get_rocstories_dataloader(batch_size, seq_len)
    
    model = TRM_Diffusion(vocab_size=vocab_size, d_model=512, num_layers=4).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)
    ema = EMA(model, decay=0.999)
    
    # Initialize TensorBoard Writer
    writer = SummaryWriter(log_dir="runs/1M_stories_diffusion")
    
    print("Starting Training with TensorBoard...")
    model.train()
    global_step = 0 
    
    for epoch in range(EPOCHS): 
        pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=True)
        
        for step, batch in enumerate(pbar):
            input_ids = batch["input_ids"].to(device)
            bs = input_ids.shape[0]
            
            y_true_emb = model.token_emb(input_ids)
            positions = torch.arange(seq_len, device=device).unsqueeze(0).expand(bs, seq_len)
            y_true_emb = y_true_emb + model.pos_emb(positions)
            
            t = torch.randint(1, T_DIFFUSION + 1, (bs,), device=device)
            alpha_bar_t = compute_sqrt_alpha_bar(t, T_max=T_DIFFUSION).view(bs, 1, 1)
            noise = torch.randn_like(y_true_emb)
            
            x_t_emb = torch.sqrt(alpha_bar_t) * y_true_emb + torch.sqrt(1 - alpha_bar_t) * noise
            
            z = model.z_init.expand(bs, seq_len, -1)
            pred_x0_emb, z, _ = model.deep_recursion(x_t_emb, z, t, n=N_RECURSIONS, T=T_CYCLES)
            
            loss_recon = F.mse_loss(pred_x0_emb, y_true_emb)
            loss_emb_anchor = F.mse_loss(y_true_emb, pred_x0_emb.detach())
            total_loss = loss_recon + (0.1 * loss_emb_anchor)
            
            optimizer.zero_grad()
            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            ema.update()
            
            # Log to TensorBoard
            writer.add_scalar("Loss/Total", total_loss.item(), global_step)
            writer.add_scalar("Loss/Reconstruction", loss_recon.item(), global_step)
            
            if step % 10 == 0:
                pbar.set_postfix({'Loss': f"{total_loss.item():.4f}"})
                
            if global_step > 0 and global_step % 10000 == 0:
                ema.apply_shadow()
                save_path = f"trm_diffusion_step_{global_step}.pt"
                torch.save({
                    'epoch': epoch,
                    'global_step': global_step,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                }, save_path)
                ema.restore()
                
            global_step += 1

        print(f"--- Epoch {epoch+1} Complete ---")
        ema.apply_shadow()
        save_path = f"trm_diffusion_epoch_{epoch+1}_complete.pt"
        torch.save({
            'epoch': epoch,
            'global_step': global_step,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
        }, save_path)
        ema.restore()
        
    writer.close()

if __name__ == "__main__":
    train()

Starting Training with TensorBoard...


Epoch 1/10: 100%|██████████| 62500/62500 [13:31:20<00:00,  1.28it/s, Loss=0.0144]  


--- Epoch 1 Complete ---


Epoch 2/10:   1%|          | 545/62500 [06:24<12:07:38,  1.42it/s, Loss=0.0163]


KeyboardInterrupt: 

In [ ]:
!tensorboard --logdir=runs

In [3]:
import torch
from datasets import load_dataset
from torch.utils.data import DataLoader
from transformers import GPT2Tokenizer

def get_rocstories_dataloader(batch_size, seq_len):
    # Load dataset from HuggingFace
    dataset = load_dataset("mintujupally/ROCStories", split='train')
    tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
    tokenizer.pad_token = tokenizer.eos_token

    def tokenize_function(examples):
        # Concatenate sentences 1-5 into a single story
        
        return tokenizer(
            examples['text'], 
            padding="max_length", 
            truncation=True, 
            max_length=seq_len, 
            return_tensors="pt"
        )

    tokenized_datasets = dataset.map(
        tokenize_function, 
        batched=True, 
        remove_columns=dataset.column_names
    )
    tokenized_datasets.set_format("torch")

    dataloader = DataLoader(tokenized_datasets, batch_size=batch_size, shuffle=True, num_workers=4)
    return dataloader, len(tokenizer)

In [4]:
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.cuda.amp import GradScaler, autocast
from transformers import GPT2Tokenizer
from datasets import load_dataset
import wandb
from tqdm import tqdm
import os

def train_roc_diffusion():
    # --- Hyperparameters & Config ---
    device = torch.device("cuda")
    seq_len = 64
    batch_size = 128  # Optimized for L40S 48GB VRAM
    epochs = 10
    lr = 1e-4
    save_dir = "/workspace/Diffusion_TRM/checkpoints" # Standard RunPod persistent path
    os.makedirs(save_dir, exist_ok=True)

    # Initialize Dataset & Model
    dataloader, vocab_size = get_rocstories_dataloader(batch_size, seq_len)
    model = TRM_Diffusion(vocab_size=vocab_size, d_model=512, num_layers=6).to(device)
    
    # Optimizer & Scaler
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    scaler = GradScaler() 
    ema = EMA(model, decay=0.9999) # Diffusion-LM requires EMA for generation quality

    # --- WandB Initialization ---
    wandb.init(
        project="diffusion-lm-roc",
        name="L40S-ROCStories-Run1",
        config={
            "batch_size": batch_size,
            "learning_rate": lr,
            "epochs": epochs,
            "architecture": "TRM-Diffusion",
            "gpu": "NVIDIA L40S"
        }
    )

    global_step = 0
    print(f"Starting Training on {device}...")

    for epoch in range(epochs):
        model.train()
        pbar = tqdm(dataloader, desc=f"Epoch {epoch}")
        
        for batch in pbar:
            input_ids = batch["input_ids"].to(device)
            
            # 1. Get Target Embeddings
            with torch.no_grad():
                y_true_emb = model.token_emb(input_ids)
                positions = torch.arange(seq_len, device=device).unsqueeze(0)
                y_true_emb = y_true_emb + model.pos_emb(positions)

            # 2. Add Diffusion Noise
            t = torch.randint(1, 2001, (input_ids.shape[0],), device=device)
            alpha_bar_t = compute_sqrt_alpha_bar(t).view(-1, 1, 1)
            noise = torch.randn_like(y_true_emb)
            x_t_emb = torch.sqrt(alpha_bar_t) * y_true_emb + torch.sqrt(1 - alpha_bar_t) * noise

            # 3. Forward Pass (Mixed Precision)
            optimizer.zero_grad()
            with autocast(dtype=torch.bfloat16):
                z = model.z_init.expand(input_ids.shape[0], seq_len, -1)
                pred_x0_emb, _, _ = model.deep_recursion(x_t_emb, z, t, n=6, T=3)
                
                # Combined Loss (MSE + Embedding Anchor)
                loss_recon = F.mse_loss(pred_x0_emb, y_true_emb)
                loss_anchor = F.mse_loss(y_true_emb, pred_x0_emb.detach())
                total_loss = loss_recon + (0.1 * loss_anchor)

            # 4. Scaled Backward Pass
            scaler.scale(total_loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            
            # Update EMA weights
            ema.update()

            # Logging
            if global_step % 20 == 0:
                wandb.log({
                    "train/loss": total_loss.item(),
                    "train/recon_loss": loss_recon.item(),
                    "train/lr": optimizer.param_groups[0]['lr'],
                    "global_step": global_step
                })
                pbar.set_postfix({"Loss": f"{total_loss.item():.4f}"})
            
            global_step += 1

        # --- END OF EPOCH SAVING ---
        print(f"Epoch {epoch} finished. Saving EMA checkpoint...")
        
        # We save the EMA weights because they are much better for sampling
        ema.apply_shadow()
        
        save_path = os.path.join(save_dir, f"diffusion_roc_epoch_{epoch}.pt")
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'global_step': global_step,
            'vocab_size': vocab_size
        }, save_path)
        
        # Important: Restore original weights so training continues naturally
        ema.restore()
        
        print(f"Saved: {save_path}")

    wandb.finish()
    print("Training Complete!")

if __name__ == "__main__":
    train_roc_diffusion()

Repo card metadata block was not found. Setting CardData to empty.
/tmp/ipykernel_5352/931836972.py:27: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: rao-adnan-098 (rao-adnan-098-jamia-millia-islamia) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Starting Training on cuda...


Epoch 0:   0%|          | 0/614 [00:00<?, ?it/s]/tmp/ipykernel_5352/931836972.py:67: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(dtype=torch.bfloat16):
Epoch 0: 100%|██████████| 614/614 [01:12<00:00,  8.51it/s, Loss=2.1655]


Epoch 0 finished. Saving EMA checkpoint...
Saved: /workspace/Diffusion_TRM/checkpoints/diffusion_roc_epoch_0.pt


Epoch 1: 100%|██████████| 614/614 [01:07<00:00,  9.06it/s, Loss=2.1520]


...

Epoch 8 finished. Saving EMA checkpoint...
Saved: /workspace/Diffusion_TRM/checkpoints/diffusion_roc_epoch_8.pt


Epoch 9: 100%|██████████| 614/614 [01:10<00:00,  8.77it/s, Loss=2.1383]


Epoch 9 finished. Saving EMA checkpoint...
Saved: /workspace/Diffusion_TRM/checkpoints/diffusion_roc_epoch_9.pt


global_step,▁▁▁▂▂▂▂▃▃▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇█
train/loss,█▅▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▁▁▂
train/lr,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/recon_loss,█▅▃▃▂▃▂▂▂▂▁▂▂▁▂▂▁▂▂▁▁▁▁▂▁▁▂▁▁▂▁▁▁▁▁▁▁▁▁▁
global_step,6120
train/loss,2.13826
train/lr,0.0001
train/recon_loss,1.94387


Training Complete!
